# Notebook 2: Run Model Evaluations (Patch Attack)

**VLM-ARB Cloud Framework**

This notebook evaluates models on patch-poisoned vs clean image pairs and logs results to Firestore.

## Workflow
1. Setup: Authenticate with Firebase and load patch dataset
2. Build clean/poisoned pairs from patch folders + labels
3. Test models (CLIP, MobileViT, BLIP-2, LLaVA, lightweight models)
4. Compute evaluation metrics (ASR, ODS, SBR)
5. Stream results to Firestore in real-time
6. Aggregate and finalize

**Key Feature**: Real-time logging to Firestore for patch attack runs.

---

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Cell 1: Install Dependencies & Setup

In [2]:
# Install pip packages for model evaluation
import subprocess
import sys

packages = [
    'firebase-admin',
    'torch',
    'torchvision',
    'transformers',
    'pillow',
    'numpy',
    'scipy',
    'scikit-learn',
    'tqdm',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("✅ All dependencies installed")

✅ All dependencies installed


## Cell 2: Setup & Load Dataset Info

import sys
sys.path.insert(0, '/root')

from pathlib import Path
from utilities.auth_helpers import quick_colab_setup, get_or_create_run_id
from utilities.cloud_sync import FirebaseSync, FirestoreMetricsLogger, validate_firebase_credentials
from datetime import datetime
import logging

# Quick setup
team_folder, context, gpu_info = quick_colab_setup()

# Initialize Firebase (optional - only for logging results)
creds_path = context['creds_path']
try:
    fs = FirebaseSync(creds_path)
    print("✅ Firebase initialized for logging results")
except:
    fs = None
    print("⚠️  Firebase unavailable - will log to local cache only")

# Load dataset info from Google Drive
drive_datasets_dir = Path("/content/drive/MyDrive/VLM-ARB-Team/datasets")
dataset_version = "drive-based"
dataset_info = {
    'base_image_count': len(list((drive_datasets_dir / "base_images").glob("*.png"))) if (drive_datasets_dir / "base_images").exists() else 0,
    'total_variants': len(list((drive_datasets_dir / "attacked_images").glob("*.png"))) if (drive_datasets_dir / "attacked_images").exists() else 0,
}

print(f"\n📦 Dataset Info:")
print(f"   Location: Google Drive (VLM-ARB-Team/datasets)")
print(f"   Base Images: {dataset_info.get('base_image_count', '?')}")
print(f"   Total Variants: {dataset_info.get('total_variants', '?')}")

# Generate unique run ID
run_id = get_or_create_run_id(team_folder, prefix="eval")
print(f"\n📊 Run ID: {run_id}")

In [ ]:
import sys
sys.path.insert(0, '/root')

from pathlib import Path
from datetime import datetime
import logging

# Ensure Drive is mounted (safe if already mounted)
from google.colab import drive
drive.mount('/content/drive')

# Ensure /root/utilities exists so imports work in Colab
colab_utilities = Path('/root/utilities')
colab_utilities.mkdir(parents=True, exist_ok=True)
(colab_utilities / '__init__.py').write_text('# Utilities module\n')

# Try to copy utilities from team Drive folder if available
project_dir = Path('/content/drive/MyDrive/VLM-ARB-Team')
source_utilities = project_dir / 'utilities'
if source_utilities.exists():
    import shutil
    shutil.copytree(source_utilities, colab_utilities, dirs_exist_ok=True)

# Create fallback cloud_sync if missing
fallback_cloud_sync_path = colab_utilities / 'cloud_sync.py'
if not fallback_cloud_sync_path.exists():
    fallback_cloud_sync = '''
from pathlib import Path
from typing import Dict, Any
import firebase_admin
from firebase_admin import credentials, firestore, storage

class FirebaseSync:
    def __init__(self, credentials_path: str, bucket_name: str = None):
        self.offline_mode = False
        self.db = None
        self.bucket = None
        try:
            cred = credentials.Certificate(credentials_path)
            project_id = cred.project_id
            resolved_bucket = bucket_name or f"{project_id}.appspot.com"
            if not firebase_admin._apps:
                firebase_admin.initialize_app(cred, {"storageBucket": resolved_bucket})
            self.db = firestore.client()
            self.bucket = storage.bucket()
        except Exception as e:
            print(f"⚠️ Firebase init fallback failed: {e}")
            self.offline_mode = True

    def upload_file(self, local_path: str, bucket_path: str, make_public: bool = False) -> bool:
        if self.offline_mode or not self.bucket:
            return False
        p = Path(local_path)
        if not p.exists():
            return False
        try:
            blob = self.bucket.blob(bucket_path)
            blob.upload_from_filename(str(p))
            if make_public:
                blob.make_public()
            return True
        except Exception as e:
            print(f"Upload failed for {local_path}: {e}")
            return False

    def upload_image_batch(self, image_dir: str, bucket_path_prefix: str):
        uploaded = {}
        p = Path(image_dir)
        if not p.exists():
            return uploaded
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.webp"):
            for img in p.glob(ext):
                bucket_path = f"{bucket_path_prefix}{img.name}"
                if self.upload_file(str(img), bucket_path, make_public=False):
                    uploaded[img.name] = bucket_path
        return uploaded

    def upload_results(self, run_id: str, metrics_dict: Dict[str, Any], metadata: Dict[str, Any] = None, collection: str = "results") -> bool:
        if self.offline_mode or not self.db:
            return False
        payload = {"metrics": metrics_dict}
        if metadata:
            payload.update(metadata)
        try:
            self.db.collection(collection).document(run_id).set(payload, merge=True)
            return True
        except Exception as e:
            print(f"Firestore upload failed: {e}")
            return False
'''
    fallback_cloud_sync_path.write_text(fallback_cloud_sync)

# Optional imports (fallback-safe)
try:
    from utilities.auth_helpers import quick_colab_setup, get_or_create_run_id
except Exception:
    quick_colab_setup = None
    get_or_create_run_id = None

# Import FirebaseSync after utilities path is guaranteed
from utilities.cloud_sync import FirebaseSync

# Team folder + context
if quick_colab_setup is not None:
    team_folder, context, gpu_info = quick_colab_setup()
else:
    team_folder = Path('/content/drive/MyDrive/VLM-ARB-Team')
    context = {'creds_path': str(team_folder / 'secrets' / 'serviceAccountKey.json'), 'user_email': 'colab_user'}
    gpu_info = {}

# Initialize Firebase (for logging only)
creds_path = Path(context['creds_path'])
try:
    fs = FirebaseSync(str(creds_path))
    if fs and not fs.offline_mode:
        print('✅ Firebase authenticated')
    else:
        print('⚠️  Firebase initialized in offline mode')
except Exception as e:
    fs = None
    print(f"⚠️  Firebase unavailable: {str(e)[:120]}")

# Drive dataset paths for patch testing
drive_datasets_dir = team_folder / 'datasets'
drive_clean_dir = drive_datasets_dir / 'patch_original' / 'images'
drive_perturbed_dir = drive_datasets_dir / 'patch_poisoned' / 'images'
drive_labels_csv = drive_datasets_dir / 'patch_poisoned' / 'labels.csv'

clean_count = len([p for p in drive_clean_dir.glob('*') if p.suffix.lower() in ['.png', '.jpg', '.jpeg', '.webp']]) if drive_clean_dir.exists() else 0
perturbed_count = len([p for p in drive_perturbed_dir.glob('*') if p.suffix.lower() in ['.png', '.jpg', '.jpeg', '.webp']]) if drive_perturbed_dir.exists() else 0

dataset_version = 'drive-patch'
dataset_info = {
    'clean_image_count': clean_count,
    'poisoned_image_count': perturbed_count,
    'labels_exists': drive_labels_csv.exists()
}

print('\n📦 Dataset Info (Patch Test):')
print(f'   Team Folder: {team_folder}')
print(f'   Clean Dir: {drive_clean_dir}')
print(f'   Poisoned Dir: {drive_perturbed_dir}')
print(f'   Labels CSV: {drive_labels_csv}')
print(f'   Clean Images: {clean_count}')
print(f'   Poisoned Images: {perturbed_count}')
print(f"   Labels Present: {dataset_info['labels_exists']}")

if get_or_create_run_id is not None:
    run_id = get_or_create_run_id(team_folder, prefix='eval_patch')
else:
    run_id = f"eval_patch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f'\n📊 Run ID: {run_id}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Firebase authenticated

📦 Dataset Info (Perturbation Test):
   Team Folder: /content/drive/MyDrive/VLM-ARB-Team
   Clean Dir: /content/drive/MyDrive/VLM-ARB-Team/datasets/Pertubation_original/images
   Perturbed Dir: /content/drive/MyDrive/VLM-ARB-Team/datasets/pertubation_pertubated/images
   Labels CSV: /content/drive/MyDrive/VLM-ARB-Team/datasets/pertubation_pertubated/labels.csv
   Clean Images: 300
   Perturbed Images: 300
   Labels Present: True

📊 Run ID: eval_perturb_20260407_173401


## Cell 3: Download Test Images from Cloud Storage

In [ ]:
import csv
from pathlib import Path
import shutil

# Source dataset paths from Cell 2
source_clean_dir = drive_clean_dir
source_perturbed_dir = drive_perturbed_dir
source_labels_csv = drive_labels_csv

# Create local evaluation directories
data_dir = Path("/tmp/vlm_arb_eval_data_patch")
clean_images_dir = data_dir / "clean_images"
perturbed_images_dir = data_dir / "poisoned_images"
local_labels_csv = data_dir / "labels.csv"

clean_images_dir.mkdir(parents=True, exist_ok=True)
perturbed_images_dir.mkdir(parents=True, exist_ok=True)

print(f"📂 Data directories: {data_dir}")

def is_image_file(p: Path) -> bool:
    return p.is_file() and p.suffix.lower() in [".png", ".jpg", ".jpeg", ".webp"]

# Copy clean images
clean_files = [p for p in source_clean_dir.glob("*") if is_image_file(p)] if source_clean_dir.exists() else []
for src_path in clean_files:
    dst_path = clean_images_dir / src_path.name
    if not dst_path.exists():
        shutil.copy2(src_path, dst_path)

# Copy poisoned images
perturbed_files = [p for p in source_perturbed_dir.glob("*") if is_image_file(p)] if source_perturbed_dir.exists() else []
for src_path in perturbed_files:
    dst_path = perturbed_images_dir / src_path.name
    if not dst_path.exists():
        shutil.copy2(src_path, dst_path)

# Copy labels if present
if source_labels_csv.exists():
    shutil.copy2(source_labels_csv, local_labels_csv)

# Build matched pairs by filename
clean_map = {p.name: p for p in clean_images_dir.glob("*") if is_image_file(p)}
pert_map = {p.name: p for p in perturbed_images_dir.glob("*") if is_image_file(p)}
pair_names = sorted(set(clean_map.keys()) & set(pert_map.keys()))

paired_samples = [
    {
        "file_name": n,
        "clean_path": str(clean_map[n]),
        "perturbed_path": str(pert_map[n])
    }
    for n in pair_names
]

print("\n✅ Dataset loaded for patch testing")
print(f"   Clean images copied: {len(clean_files)}")
print(f"   Poisoned images copied: {len(perturbed_files)}")
print(f"   Matched pairs: {len(paired_samples)}")
print(f"   Labels copied: {'yes' if local_labels_csv.exists() else 'no'}")

if len(paired_samples) == 0:
    print("\n⚠️  No matched clean/poisoned pairs found by filename.")
    print("   Ensure both folders contain files with the same names.")

📂 Data directories: /tmp/vlm_arb_eval_data

✅ Dataset loaded for perturbation testing
   Clean images copied: 300
   Perturbed images copied: 300
   Matched pairs: 300
   Labels copied: yes


## Cell 4: Load Models (with Graceful Degradation)

In [5]:
import torch
from PIL import Image
import numpy as np

# Check available GPU memory
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"💾 GPU Memory: {gpu_memory:.1f} GB")
else:
    print("⚠️  No GPU available. Using CPU (slow)")
    gpu_memory = 0

models_to_load = {}
models_status = {}

# ===== CLIP =====
try:
    from transformers import CLIPProcessor, CLIPModel
    print("\n📦 Loading CLIP...")
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    if torch.cuda.is_available():
        clip_model = clip_model.cuda()
    clip_model.eval()
    models_to_load['clip'] = (clip_model, clip_processor)
    models_status['clip'] = "✅ Loaded"
    print("   ✅ CLIP loaded")
except Exception as e:
    models_status['clip'] = f"❌ Failed: {str(e)[:50]}"
    print(f"   ❌ CLIP failed: {e}")

# ===== MobileViT (lightweight classifier) =====
try:
    from transformers import MobileViTImageProcessor, MobileViTForImageClassification
    print("\n📦 Loading MobileViT...")
    mobilevit_processor = MobileViTImageProcessor.from_pretrained("apple/mobilevit-small")
    mobilevit_model = MobileViTForImageClassification.from_pretrained("apple/mobilevit-small")
    if torch.cuda.is_available():
        mobilevit_model = mobilevit_model.cuda()
    mobilevit_model.eval()
    models_to_load['mobilevit'] = (mobilevit_model, mobilevit_processor)
    models_status['mobilevit'] = "✅ Loaded"
    print("   ✅ MobileViT loaded")
except Exception as e:
    models_status['mobilevit'] = f"❌ Failed: {str(e)[:50]}"
    print(f"   ❌ MobileViT failed: {e}")

# ===== BLIP-2 (large, may fail) =====
try:
    if gpu_memory > 10:  # Need ~10GB
        from transformers import AutoProcessor, Blip2ForConditionalGeneration
        print("\n📦 Loading BLIP-2...")
        blip2_processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
        blip2_model = Blip2ForConditionalGeneration.from_pretrained(
            "Salesforce/blip2-opt-2.7b",
            torch_dtype=torch.float16
        )
        if torch.cuda.is_available():
            blip2_model = blip2_model.cuda()
        blip2_model.eval()
        models_to_load['blip2'] = (blip2_model, blip2_processor)
        models_status['blip2'] = "✅ Loaded"
        print("   ✅ BLIP-2 loaded")
    else:
        models_status['blip2'] = "⏭️ Skipped (insufficient GPU memory)"
        print(f"   ⏭️  Skipping BLIP-2 (GPU memory: {gpu_memory:.1f}GB < 10GB required)")
except Exception as e:
    models_status['blip2'] = f"❌ Failed: {str(e)[:50]}"
    print(f"   ❌ BLIP-2 failed: {e}")

# ===== LLaVA (very large, likely to fail) =====
try:
    if gpu_memory > 12:  # Need ~14GB
        print("\n📦 Loading LLaVA...")
        from transformers import AutoTokenizer, LlavaForConditionalGeneration
        llava_processor = AutoTokenizer.from_pretrained("llava-hf/llava-1.5-7b-hf")
        llava_model = LlavaForConditionalGeneration.from_pretrained(
            "llava-hf/llava-1.5-7b-hf",
            torch_dtype=torch.float16
        )
        if torch.cuda.is_available():
            llava_model = llava_model.cuda()
        llava_model.eval()
        models_to_load['llava'] = (llava_model, llava_processor)
        models_status['llava'] = "✅ Loaded"
        print("   ✅ LLaVA loaded")
    else:
        models_status['llava'] = "⏭️ Skipped (insufficient GPU memory)"
        print(f"   ⏭️  Skipping LLaVA (GPU memory: {gpu_memory:.1f}GB < 14GB required)")
except Exception as e:
    models_status['llava'] = f"❌ Failed: {str(e)[:50]}"
    print(f"   ❌ LLaVA failed: {e}")


# ===== Additional Lightweight Models =====
from transformers import AutoImageProcessor, AutoModelForImageClassification

# 1. ViT-Tiny
try:
    print("\n📦 Loading ViT-Tiny...")
    vit_tiny_processor = AutoImageProcessor.from_pretrained("google/vit-tiny-patch16-224")
    vit_tiny_model = AutoModelForImageClassification.from_pretrained("google/vit-tiny-patch16-224")
    if torch.cuda.is_available():
        vit_tiny_model = vit_tiny_model.cuda()
    vit_tiny_model.eval()
    models_to_load['vit_tiny'] = (vit_tiny_model, vit_tiny_processor)
    models_status['vit_tiny'] = "✅ Loaded"
    print("   ✅ ViT-Tiny loaded")
except Exception as e:
    models_status['vit_tiny'] = f"❌ Failed: {str(e)[:50]}"
    print(f"   ❌ ViT-Tiny failed: {e}")

# 2. DeiT-Tiny
try:
    print("\n📦 Loading DeiT-Tiny...")
    deit_tiny_processor = AutoImageProcessor.from_pretrained("facebook/deit-tiny-patch16-224")
    deit_tiny_model = AutoModelForImageClassification.from_pretrained("facebook/deit-tiny-patch16-224")
    if torch.cuda.is_available():
        deit_tiny_model = deit_tiny_model.cuda()
    deit_tiny_model.eval()
    models_to_load['deit_tiny'] = (deit_tiny_model, deit_tiny_processor)
    models_status['deit_tiny'] = "✅ Loaded"
    print("   ✅ DeiT-Tiny loaded")
except Exception as e:
    models_status['deit_tiny'] = f"❌ Failed: {str(e)[:50]}"
    print(f"   ❌ DeiT-Tiny failed: {e}")

# 3. MobileNetV2
try:
    print("\n📦 Loading MobileNetV2...")
    mobilenet_processor = AutoImageProcessor.from_pretrained("google/mobilenet_v2_1.0_224")
    mobilenet_model = AutoModelForImageClassification.from_pretrained("google/mobilenet_v2_1.0_224")
    if torch.cuda.is_available():
        mobilenet_model = mobilenet_model.cuda()
    mobilenet_model.eval()
    models_to_load['mobilenet'] = (mobilenet_model, mobilenet_processor)
    models_status['mobilenet'] = "✅ Loaded"
    print("   ✅ MobileNetV2 loaded")
except Exception as e:
    models_status['mobilenet'] = f"❌ Failed: {str(e)[:50]}"
    print(f"   ❌ MobileNetV2 failed: {e}")

# 4. BEiT-Base
try:
    print("\n📦 Loading BEiT-Base...")
    beit_processor = AutoImageProcessor.from_pretrained("microsoft/beit-base-patch16-224-pt1k")
    beit_model = AutoModelForImageClassification.from_pretrained("microsoft/beit-base-patch16-224-pt1k")
    if torch.cuda.is_available():
        beit_model = beit_model.cuda()
    beit_model.eval()
    models_to_load['beit_base'] = (beit_model, beit_processor)
    models_status['beit_base'] = "✅ Loaded"
    print("   ✅ BEiT-Base loaded")
except Exception as e:
    models_status['beit_base'] = f"❌ Failed: {str(e)[:50]}"
    print(f"   ❌ BEiT-Base failed: {e}")

# 5. ResNet-18 (from Hugging Face timm collection)
try:
    print("\n📦 Loading ResNet-18...")
    resnet_processor = AutoImageProcessor.from_pretrained("timm/resnet18.a1_in1k")
    resnet_model = AutoModelForImageClassification.from_pretrained("timm/resnet18.a1_in1k")
    if torch.cuda.is_available():
        resnet_model = resnet_model.cuda()
    resnet_model.eval()
    models_to_load['resnet18'] = (resnet_model, resnet_processor)
    models_status['resnet18'] = "✅ Loaded"
    print("   ✅ ResNet-18 loaded")
except Exception as e:
    models_status['resnet18'] = f"❌ Failed: {str(e)[:50]}"
    print(f"   ❌ ResNet-18 failed: {e}")

print(f"\n📊 Model Status:")
for model_name, status in models_status.items():
    print(f"   {model_name}: {status}")


⚠️  No GPU available. Using CPU (slow)

📦 Loading CLIP...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


   ✅ CLIP loaded

📦 Loading MobileViT...


...

   ✅ MobileNetV2 loaded

📦 Loading BEiT-Base...
   ❌ BEiT-Base failed: microsoft/beit-base-patch16-224-pt1k is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

📦 Loading ResNet-18...


config.json:   0%|          | 0.00/755 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/122 [00:00<?, ?it/s]

   ✅ ResNet-18 loaded

📊 Model Status:
   clip: ✅ Loaded
   mobilevit: ✅ Loaded
   blip2: ⏭️ Skipped (insufficient GPU memory)
   llava: ⏭️ Skipped (insufficient GPU memory)
   vit_tiny: ❌ Failed: google/vit-tiny-patch16-224 is not a local folder 
   deit_tiny: ✅ Loaded
   mobilenet: ✅ Loaded
   beit_base: ❌ Failed: microsoft/beit-base-patch16-224-pt1k is not a loca
   resnet18: ✅ Loaded


## Cell 5: Run Inference (Clean + Patch-Poisoned Pairs)

In [ ]:
from pathlib import Path
from PIL import Image
import torch
import csv
from tqdm import tqdm

# Notebook 1 patch path convention:
# /content/drive/MyDrive/VLM-ARB-Team/datasets/patch_original/images
# /content/drive/MyDrive/VLM-ARB-Team/datasets/patch_poisoned/images

def _is_image_file(p: Path) -> bool:
    return p.is_file() and p.suffix.lower() in ['.png', '.jpg', '.jpeg', '.webp']

def _build_pairs_robustly(clean_dir_path: Path, pert_dir_path: Path, labels_csv_path: Path, max_pairs: int = 50):
    clean_map = {p.name: p for p in clean_dir_path.glob('*') if _is_image_file(p)} if clean_dir_path.exists() else {}
    pert_map = {p.name: p for p in pert_dir_path.glob('*') if _is_image_file(p)} if pert_dir_path.exists() else {}

    pair_names = sorted(set(clean_map.keys()) & set(pert_map.keys()))
    pairs = [
        {'file_name': n, 'clean_path': str(clean_map[n]), 'perturbed_path': str(pert_map[n])}
        for n in pair_names
    ]

    if not pairs and labels_csv_path.exists():
        try:
            rows = list(csv.DictReader(labels_csv_path.open('r', encoding='utf-8')))
            clean_keys = ['clean', 'clean_file', 'clean_filename', 'original', 'original_file', 'image_file', 'filename']
            pert_keys = ['perturbed', 'poison_file', 'poisoned', 'attacked', 'file_name', 'image_file', 'filename']

            for row in rows:
                clean_name = None
                pert_name = None

                for k in clean_keys:
                    if k in row and row[k]:
                        clean_name = Path(row[k]).name
                        break
                for k in pert_keys:
                    if k in row and row[k]:
                        pert_name = Path(row[k]).name
                        break

                if clean_name in clean_map and pert_name in pert_map:
                    pairs.append({
                        'file_name': pert_name,
                        'clean_path': str(clean_map[clean_name]),
                        'perturbed_path': str(pert_map[pert_name]),
                    })
        except Exception as e:
            print(f"⚠️ labels.csv parsing failed in Cell 5 (fallback): {str(e)[:120]}")

    unique = []
    seen = set()
    for p in pairs:
        key = (p['clean_path'], p['perturbed_path'])
        if key not in seen:
            seen.add(key)
            unique.append(p)

    return unique[:max_pairs], len(clean_map), len(pert_map)

MAX_EVAL_PAIRS = 300

clean_dir_path = Path('/content/drive/MyDrive/VLM-ARB-Team/datasets/patch_original/images')
pert_dir_path = Path('/content/drive/MyDrive/VLM-ARB-Team/datasets/patch_poisoned/images')
labels_csv_path = Path('/content/drive/MyDrive/VLM-ARB-Team/datasets/patch_poisoned/labels.csv')

paired_samples, clean_n, pert_n = _build_pairs_robustly(clean_dir_path, pert_dir_path, labels_csv_path, max_pairs=MAX_EVAL_PAIRS)

print(f"\n✅ Prepared common paired samples (limited to {MAX_EVAL_PAIRS} pairs):")
print(f"   Clean images found: {clean_n}")
print(f"   Patch-poisoned images found: {pert_n}")
print(f"   labels.csv exists: {labels_csv_path.exists()}")
print(f"   Matched pairs: {len(paired_samples)}")

# ===== CLIP Evaluation =====
if 'clip' in models_to_load:
    print("\n🧠 Testing CLIP on patch pairs...")

    clip_model, clip_processor = models_to_load['clip']
    candidate_labels = ["a photo", "text", "object", "scene", "abstract"]

    def _clip_predict(image_path: str):
        img = Image.open(image_path).convert('RGB')
        with torch.no_grad():
            inputs = clip_processor(
                text=candidate_labels,
                images=img,
                return_tensors="pt",
                padding=True,
            )
            if torch.cuda.is_available():
                inputs = {k: v.cuda() for k, v in inputs.items()}

            outputs = clip_model(**inputs)
            probs = outputs.logits_per_image.softmax(dim=1)

        top_idx = probs.argmax().item()
        return candidate_labels[top_idx], float(probs[0, top_idx])

    clip_pair_results = []
    if paired_samples:
        for sample in tqdm(paired_samples, desc="CLIP clean vs patch-poisoned"):
            try:
                clean_label, clean_conf = _clip_predict(sample['clean_path'])
                pert_label, pert_conf = _clip_predict(sample['perturbed_path'])

                clip_pair_results.append(
                    {
                        'file_name': sample['file_name'],
                        'clean_label': clean_label,
                        'clean_conf': clean_conf,
                        'perturbed_label': pert_label,
                        'perturbed_conf': pert_conf,
                        'changed': clean_label != pert_label,
                    }
                )
            except Exception as e:
                print(f"   Error processing pair {sample.get('file_name', '<unknown>')}: {e}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        changed_n = sum(1 for r in clip_pair_results if r['changed'])
        total_n = len(clip_pair_results)
        change_rate = (changed_n / total_n) if total_n else 0.0

        print(f"\n✅ CLIP: Processed {total_n} clean/patch-poisoned pairs")
        print(f"   Label changes: {changed_n}/{total_n} ({change_rate:.1%})")
    else:
        print("\n⚠️ No paired samples available to evaluate with CLIP.")

else:
    print("⏭️  CLIP not loaded - skipping")
    clip_pair_results = []

# ===== MobileViT Evaluation =====
mobilevit_pair_results = []
if 'mobilevit' in models_to_load:
    print("\n🧠 Testing MobileViT on patch pairs...")

    mobilevit_model, mobilevit_processor = models_to_load['mobilevit']

    def _mobilevit_predict(image_path: str):
        img = Image.open(image_path).convert('RGB')
        with torch.no_grad():
            inputs = mobilevit_processor(images=img, return_tensors="pt")
            if torch.cuda.is_available():
                inputs = {k: v.cuda() for k, v in inputs.items()}

            outputs = mobilevit_model(**inputs)
            logits = outputs.logits
            predicted_class_idx = logits.argmax(-1).item()

            predicted_label = mobilevit_model.config.id2label.get(predicted_class_idx, str(predicted_class_idx))
            probabilities = torch.nn.functional.softmax(logits, dim=-1)[0]
            confidence = probabilities[predicted_class_idx].item()

        return predicted_label, confidence

    if paired_samples:
        for sample in tqdm(paired_samples, desc="MobileViT clean vs patch-poisoned"):
            try:
                clean_label, clean_conf = _mobilevit_predict(sample['clean_path'])
                pert_label, pert_conf = _mobilevit_predict(sample['perturbed_path'])

                mobilevit_pair_results.append(
                    {
                        'file_name': sample['file_name'],
                        'clean_label': clean_label,
                        'clean_conf': clean_conf,
                        'perturbed_label': pert_label,
                        'perturbed_conf': pert_conf,
                        'changed': clean_label != pert_label,
                    }
                )
            except Exception as e:
                print(f"   Error processing pair {sample.get('file_name', '<unknown>')}: {e}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        changed_n = sum(1 for r in mobilevit_pair_results if r['changed'])
        total_n = len(mobilevit_pair_results)
        change_rate = (changed_n / total_n) if total_n else 0.0

        print(f"\n✅ MobileViT: Processed {total_n} clean/patch-poisoned pairs")
        print(f"   Label changes: {changed_n}/{total_n} ({change_rate:.1%})")
    else:
        print("\n⚠️ No paired samples available to evaluate with MobileViT.")

else:
    print("⏭️  MobileViT not loaded - skipping")

# ===== BLIP-2 Evaluation (placeholder) =====
blip2_pair_results = []
if 'blip2' in models_to_load:
    print("\n🧠 Testing BLIP-2 on patch pairs... (not yet implemented)")
    print("⏭️  BLIP-2 evaluation logic not implemented for this patch test.")
else:
    print("⏭️  BLIP-2 not loaded - skipping evaluation")

# ===== LLaVA Evaluation (placeholder) =====
llava_pair_results = []
if 'llava' in models_to_load:
    print("\n🧠 Testing LLaVA on patch pairs... (not yet implemented)")
    print("⏭️  LLaVA evaluation logic not implemented for this patch test.")
else:
    print("⏭️  LLaVA not loaded - skipping evaluation")

# ===== Generic image classifier evaluation =====
def _classify_predict(model, processor, image_path: str):
    img = Image.open(image_path).convert('RGB')
    with torch.no_grad():
        inputs = processor(images=img, return_tensors="pt")
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}

        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class_idx = logits.argmax(-1).item()
        predicted_label = model.config.id2label.get(predicted_class_idx, str(predicted_class_idx))
        probabilities = torch.nn.functional.softmax(logits, dim=-1)[0]
        confidence = probabilities[predicted_class_idx].item()
    return predicted_label, confidence

new_lightweight_models = ['vit_tiny', 'deit_tiny', 'mobilenet', 'beit_base', 'resnet18']
all_new_results = {}

for model_name in new_lightweight_models:
    model_pair_results = []
    if model_name in models_to_load:
        print(f"\n🧠 Testing {model_name} on patch pairs...")
        model, processor = models_to_load[model_name]

        if paired_samples:
            for sample in tqdm(paired_samples, desc=f"{model_name} clean vs patch-poisoned"):
                try:
                    clean_label, clean_conf = _classify_predict(model, processor, sample['clean_path'])
                    pert_label, pert_conf = _classify_predict(model, processor, sample['perturbed_path'])

                    model_pair_results.append(
                        {
                            'file_name': sample['file_name'],
                            'clean_label': clean_label,
                            'clean_conf': clean_conf,
                            'perturbed_label': pert_label,
                            'perturbed_conf': pert_conf,
                            'changed': clean_label != pert_label,
                        }
                    )
                except Exception as e:
                    print(f"   Error processing pair {sample.get('file_name', '<unknown>')} for {model_name}: {e}")
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            changed_n = sum(1 for r in model_pair_results if r['changed'])
            total_n = len(model_pair_results)
            change_rate = (changed_n / total_n) if total_n else 0.0

            print(f"\n✅ {model_name}: Processed {total_n} clean/patch-poisoned pairs")
            print(f"   Label changes: {changed_n}/{total_n} ({change_rate:.1%})")
        else:
            print(f"\n⚠️ No paired samples available to evaluate with {model_name}.")
    else:
        print(f"⏭️  {model_name} not loaded - skipping")
    all_new_results[model_name] = model_pair_results

vit_tiny_pair_results = all_new_results.get('vit_tiny', [])
deit_tiny_pair_results = all_new_results.get('deit_tiny', [])
mobilenet_pair_results = all_new_results.get('mobilenet', [])
beit_base_pair_results = all_new_results.get('beit_base', [])
resnet18_pair_results = all_new_results.get('resnet18', [])


✅ Prepared common paired samples (limited to 300 pairs):
   Clean images found: 300
   Perturbed images found: 300
   labels.csv exists: True
   Matched pairs: 300

🧠 Testing CLIP on perturbation pairs...


CLIP clean vs perturbed: 100%|██████████| 300/300 [03:30<00:00,  1.42it/s]



✅ CLIP: Processed 300 clean/perturbed pairs
   Label changes: 9/300 (3.0%)

🧠 Testing MobileViT on perturbation pairs...


MobileViT clean vs perturbed: 100%|██████████| 300/300 [01:51<00:00,  2.70it/s]



✅ MobileViT: Processed 300 clean/perturbed pairs
   Label changes: 17/300 (5.7%)
⏭️  BLIP-2 not loaded - skipping evaluation
⏭️  LLaVA not loaded - skipping evaluation
⏭️  vit_tiny not loaded - skipping

🧠 Testing deit_tiny on perturbation pairs...


...


✅ deit_tiny: Processed 300 clean/perturbed pairs
   Label changes: 38/300 (12.7%)

🧠 Testing mobilenet on perturbation pairs...


mobilenet clean vs perturbed: 100%|██████████| 300/300 [00:37<00:00,  7.95it/s]



✅ mobilenet: Processed 300 clean/perturbed pairs
   Label changes: 32/300 (10.7%)
⏭️  beit_base not loaded - skipping

🧠 Testing resnet18 on perturbation pairs...


resnet18 clean vs perturbed: 100%|██████████| 300/300 [01:17<00:00,  3.89it/s]


✅ resnet18: Processed 300 clean/perturbed pairs
   Label changes: 43/300 (14.3%)


In [11]:
import json

output_filename = f"{run_id}_metrics.json"
with open(output_filename, 'w') as f:
    json.dump(metrics_payload, f, indent=4)

print(f"✅ Results saved to {output_filename}")

✅ Results saved to eval_perturb_20260407_173401_metrics.json


## Cell 6: Compute Evaluation Metrics (ASR, ODS, SBR)

In [7]:
from scipy.spatial.distance import cosine

def compute_asr(clean_output, attacked_output):
    """
    Attack Success Rate: Did output change?
    Returns: 1.0 if changed, 0.0 if same
    """
    if isinstance(clean_output, str) and isinstance(attacked_output, str):
        return 1.0 if clean_output != attacked_output else 0.0
    return 0.0

def compute_ods(clean_output, attacked_output):
    """
    Output Deviation Score: How different is the output?
    For now: binary (changed=0.5, unchanged=0.0)
    """
    if isinstance(clean_output, str) and isinstance(attacked_output, str):
        if clean_output != attacked_output:
            return 0.5  # Simplified: moderate deviation
    return 0.0

def compute_sbr(output_text):
    """
    Safety Bypass Rate: Did it ignore safety guidelines?
    Check for keywords suggesting safety bypass
    """
    if isinstance(output_text, str):
        suspicious = ["ignore", "bypass", "unsafe", "violation"]
        if any(word in output_text.lower() for word in suspicious):
            return 1.0
    return 0.0

print("✅ Evaluation metrics defined")

✅ Evaluation metrics defined


## Cell 7: Aggregate Metrics & Log to Firestore

In [ ]:
import csv
import numpy as np
from pathlib import Path
from PIL import Image
import torch
from datetime import datetime

# Safe defaults so this cell can run standalone
fs = globals().get('fs', None)
run_id = globals().get('run_id', f"eval_patch_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
dataset_version = globals().get('dataset_version', 'drive-patch')
models_to_load = globals().get('models_to_load', {})
pair_names = globals().get('pair_names', [])
paired_samples = globals().get('paired_samples', [])

clip_pair_results = globals().get('clip_pair_results', [])
mobilevit_pair_results = globals().get('mobilevit_pair_results', [])
blip2_pair_results = globals().get('blip2_pair_results', [])
llava_pair_results = globals().get('llava_pair_results', [])
vit_tiny_pair_results = globals().get('vit_tiny_pair_results', [])
deit_tiny_pair_results = globals().get('deit_tiny_pair_results', [])
mobilenet_pair_results = globals().get('mobilenet_pair_results', [])
beit_base_pair_results = globals().get('beit_base_pair_results', [])
resnet18_pair_results = globals().get('resnet18_pair_results', [])

class _NoOpMetricsLogger:
    def __init__(self):
        self.buffer = {}
    def log_model_metrics(self, model_name, asr=0.0, ods=0.0, sbr=0.0):
        self.buffer[model_name] = {'asr': asr, 'ods': ods, 'sbr': sbr}
    def flush(self):
        print("No-Op logger flushed.")
        return False

if fs is None:
    cred_candidates = [
        Path('/content/drive/MyDrive/VLM-ARB-Team/secrets/serviceAccountKey.json'),
        Path('/content/drive/MyDrive/drive-download-20260406T205055Z-1-001/VLM-ARB-Team/secrets/serviceAccountKey.json'),
    ]
    cred_path = next((p for p in cred_candidates if p.exists()), None)
    if cred_path is not None:
        try:
            from utilities.cloud_sync import FirebaseSync
            fs = FirebaseSync(str(cred_path))
            print(f"✅ Firebase initialized from {cred_path}")
        except Exception as e:
            print(f"⚠️ Firebase init attempt failed: {str(e)[:120]}")
    else:
        print("⚠️ No Firebase credentials found. Running in offline mode.")

if not paired_samples:
    print("⚠️ No paired samples available. Cannot compute metrics for any model.")

try:
    from utilities.cloud_sync import FirestoreMetricsLogger
    metrics_logger = FirestoreMetricsLogger(fs, run_id) if fs else _NoOpMetricsLogger()
except Exception:
    metrics_logger = _NoOpMetricsLogger()
    print("⚠️ Could not initialize FirestoreMetricsLogger. Using No-Op logger.")

print('\n📊 Computing Aggregated Metrics...')
metrics_payload = {}

if clip_pair_results:
    if len(clip_pair_results) > 0:
        change_rate = sum(1 for r in clip_pair_results if r['changed']) / len(clip_pair_results)
        conf_drop = [max(0.0, r['clean_conf'] - r['perturbed_conf']) for r in clip_pair_results]
        avg_conf_drop = float(np.mean(conf_drop)) if conf_drop else 0.0

        clip_asr = float(change_rate)
        clip_ods = float(avg_conf_drop)
        clip_sbr = 0.0

        metrics_logger.log_model_metrics('clip', asr=clip_asr, ods=clip_ods, sbr=clip_sbr)
        metrics_payload['clip'] = {'asr': clip_asr, 'ods': clip_ods, 'sbr': clip_sbr, 'pairs': len(clip_pair_results)}
        print(f"   CLIP: ASR={clip_asr:.3f}, ODS={clip_ods:.3f}, SBR={clip_sbr:.3f} (pairs={len(clip_pair_results)})")
    else:
        print('   CLIP pair results available but empty.')
else:
    print('   No CLIP pair results available')

if mobilevit_pair_results:
    if len(mobilevit_pair_results) > 0:
        change_rate = sum(1 for r in mobilevit_pair_results if r['changed']) / len(mobilevit_pair_results)
        conf_drop = [max(0.0, r['clean_conf'] - r['perturbed_conf']) for r in mobilevit_pair_results]
        avg_conf_drop = float(np.mean(conf_drop)) if conf_drop else 0.0

        mobilevit_asr = float(change_rate)
        mobilevit_ods = float(avg_conf_drop)
        mobilevit_sbr = 0.0

        metrics_logger.log_model_metrics('mobilevit', asr=mobilevit_asr, ods=mobilevit_ods, sbr=mobilevit_sbr)
        metrics_payload['mobilevit'] = {'asr': mobilevit_asr, 'ods': mobilevit_ods, 'sbr': mobilevit_sbr, 'pairs': len(mobilevit_pair_results)}
        print(f"   MobileViT: ASR={mobilevit_asr:.3f}, ODS={mobilevit_ods:.3f}, SBR={mobilevit_sbr:.3f} (pairs={len(mobilevit_pair_results)})")
    else:
        print('   MobileViT pair results available but empty.')
else:
    print('   No MobileViT pair results available')

if 'blip2' in models_to_load and blip2_pair_results:
    blip2_asr = 0.0
    blip2_ods = 0.0
    blip2_sbr = 0.0
    metrics_logger.log_model_metrics('blip2', asr=blip2_asr, ods=blip2_ods, sbr=blip2_sbr)
    metrics_payload['blip2'] = {'asr': blip2_asr, 'ods': blip2_ods, 'sbr': blip2_sbr, 'pairs': len(blip2_pair_results)}
    print(f"   BLIP-2: ASR={blip2_asr:.3f}, ODS={blip2_ods:.3f}, SBR={blip2_sbr:.3f} (pairs={len(blip2_pair_results)}) [Placeholder]")
elif 'blip2' in models_to_load:
    metrics_logger.log_model_metrics('blip2', asr=0.0, ods=0.0, sbr=0.0)
    metrics_payload['blip2'] = {'asr': 0.0, 'ods': 0.0, 'sbr': 0.0, 'pairs': 0}
    print("   BLIP-2: Placeholder metrics ASR=0.000, ODS=0.000, SBR=0.000")
else:
    print('   BLIP-2 not loaded - skipping metrics')

if 'llava' in models_to_load and llava_pair_results:
    llava_asr = 0.0
    llava_ods = 0.0
    llava_sbr = 0.0
    metrics_logger.log_model_metrics('llava', asr=llava_asr, ods=llava_ods, sbr=llava_sbr)
    metrics_payload['llava'] = {'asr': llava_asr, 'ods': llava_ods, 'sbr': llava_sbr, 'pairs': len(llava_pair_results)}
    print(f"   LLaVA: ASR={llava_asr:.3f}, ODS={llava_ods:.3f}, SBR={llava_sbr:.3f} (pairs={len(llava_pair_results)}) [Placeholder]")
elif 'llava' in models_to_load:
    metrics_logger.log_model_metrics('llava', asr=0.0, ods=0.0, sbr=0.0)
    metrics_payload['llava'] = {'asr': 0.0, 'ods': 0.0, 'sbr': 0.0, 'pairs': 0}
    print("   LLaVA: Placeholder metrics ASR=0.000, ODS=0.000, SBR=0.000")
else:
    print('   LLaVA not loaded - skipping metrics')

for model_name, results_list in [
    ('vit_tiny', vit_tiny_pair_results),
    ('deit_tiny', deit_tiny_pair_results),
    ('mobilenet', mobilenet_pair_results),
    ('beit_base', beit_base_pair_results),
    ('resnet18', resnet18_pair_results)
]:
    if results_list:
        if len(results_list) > 0:
            change_rate = sum(1 for r in results_list if r['changed']) / len(results_list)
            conf_drop = [max(0.0, r['clean_conf'] - r['perturbed_conf']) for r in results_list]
            avg_conf_drop = float(np.mean(conf_drop)) if conf_drop else 0.0

            asr = float(change_rate)
            ods = float(avg_conf_drop)
            sbr = 0.0

            metrics_logger.log_model_metrics(model_name, asr=asr, ods=ods, sbr=sbr)
            metrics_payload[model_name] = {'asr': asr, 'ods': ods, 'sbr': sbr, 'pairs': len(results_list)}
            print(f"   {model_name}: ASR={asr:.3f}, ODS={ods:.3f}, SBR={sbr:.3f} (pairs={len(results_list)})\n")
        else:
            print(f'   {model_name} pair results available but empty.\n')
    else:
        print(f'   No {model_name} pair results available\n')

print('\n☁️  Uploading results to Firestore...')
success = metrics_logger.flush()

if (not success) and fs and not getattr(fs, 'offline_mode', True) and metrics_payload:
    try:
        success = fs.upload_results(
            run_id=run_id,
            metrics_dict={
                'dataset_version': dataset_version,
                'results': metrics_payload
            },
            metadata={
                'status': 'completed',
                'mode': 'patch',
                'timestamp': datetime.now().isoformat()
            },
            collection='results'
        )
        if success:
            print("✅ Direct upload fallback successful.")
    except Exception as e:
        print(f"   Direct upload fallback failed: {str(e)[:120]}")

if success:
    print(f"✅ Results saved: {run_id}")
else:
    print('⚠️  Firestore unavailable or logger disabled. Run data kept locally in notebook output.')

⚠️ Could not initialize FirestoreMetricsLogger. Using No-Op logger.

📊 Computing Aggregated Metrics...
   CLIP: ASR=0.030, ODS=0.063, SBR=0.000 (pairs=300)
   MobileViT: ASR=0.057, ODS=0.048, SBR=0.000 (pairs=300)
   BLIP-2 not loaded - skipping metrics
   LLaVA not loaded - skipping metrics
   No vit_tiny pair results available

   deit_tiny: ASR=0.127, ODS=0.015, SBR=0.000 (pairs=300)

   mobilenet: ASR=0.107, ODS=0.064, SBR=0.000 (pairs=300)

   No beit_base pair results available

   resnet18: ASR=0.143, ODS=0.070, SBR=0.000 (pairs=300)


☁️  Uploading results to Firestore...
No-Op logger flushed.
✅ Direct upload fallback successful.
✅ Results saved: eval_perturb_20260407_173401


## Cell 8: Cleanup & Summary

In [ ]:
import shutil

print("\n" + "="*60)
print("✅ PATCH MODEL EVALUATION COMPLETE")
print("="*60)

print(f"\n📊 Results:")
print(f"   Run ID: {run_id}")
print(f"   Dataset Version: {dataset_version}")
print(f"   Models Tested: {len(models_to_load)}")
print(f"   Matched Patch Pairs Evaluated: {len(clip_pair_results) if 'clip_pair_results' in globals() else 0}")
print(f"   Status: Firestore {'✅' if fs and not fs.offline_mode else '⚠️'}")

print(f"\n📋 Next Steps:")
print(f"   1. Run Notebook 3: Generate Reports")
print(f"   2. Review patch attack metrics (ASR/ODS)")
print(f"   3. Share reports with team")

print(f"\n🧹 Cleanup (optional): Delete {data_dir}")
# shutil.rmtree(data_dir, ignore_errors=True)  # Uncomment to auto-delete

print("="*60)


✅ MODEL EVALUATION COMPLETE

📊 Results:
   Run ID: eval_perturb_20260407_173401
   Dataset Version: drive-perturbation
   Models Tested: 5
   Matched Pairs Evaluated: 300
   Status: Firestore ✅

📋 Next Steps:
   1. Run Notebook 3: Generate Reports
   2. Review perturbation metrics for CLIP (ASR/ODS)
   3. Share reports with team

🧹 Cleanup (optional): Delete /tmp/vlm_arb_eval_data


In [10]:
display(metrics_payload)

{'clip': {'asr': 0.03, 'ods': 0.06260428269704182, 'sbr': 0.0, 'pairs': 300},
 'mobilevit': {'asr': 0.056666666666666664,
  'ods': 0.047770491937796276,
  'sbr': 0.0,
  'pairs': 300},
 'deit_tiny': {'asr': 0.12666666666666668,
  'ods': 0.014722000590215127,
  'sbr': 0.0,
  'pairs': 300},
 'mobilenet': {'asr': 0.10666666666666667,
  'ods': 0.06410422546168168,
  'sbr': 0.0,
  'pairs': 300},
 'resnet18': {'asr': 0.14333333333333334,
  'ods': 0.06979695727427801,
  'sbr': 0.0,
  'pairs': 300}}